In [56]:
#imports
import pandas as pd
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

Candidates:

'Cotrim_Figueiredo'
'Gouveia_Melo'
'Filipe'
'Marques_Mendes'
'Martins'
'Ventura'
'Pinto'
'Seguro'

In [57]:
# Select candidates 

person1 = 'Ventura'
person2 = 'Filipe'

## Load visual data

In [58]:
# Load visual pkl for debate with person 1 and person 2 OR all debates with person 1 OR all debates
pklfiles = []

if not person1:
    for file in os.listdir('Project_Features'):
        if file.endswith('visual.pkl'):
            pklfiles.append(file)
elif not person2:
    for file in os.listdir('Project_Features'):
        if file.endswith('visual.pkl') and person1 in file:
            pklfiles.append(file)
else:
    for file in os.listdir('Project_Features'):
        if file.endswith('visual.pkl') and person1 in file and person2 in file:
            pklfiles.append(file)

#print(pklfiles)

data = []

for video in pklfiles:
    print(f"Loading: {video}")
    df = pd.read_pickle(os.path.join('Project_Features', video))
    data.append(df)

data = pd.concat(data, ignore_index=True)


Loading: Ventura_vs_Filipe_December_13_visual.pkl


Poses collumn:
- List of dictionaries (one for each detected person). 
- Dictionaries keys: 
    - 'bbox': A list of 4 numbers [x1, y1, x2, y2] representing the corners of the box drawn around the person's body.  
    - 'confidence': The class of the detected object (human).  
    - 'class': How sure the detection model is that it found a person. 
    - 'pose': A massive 17x3 matrix containing the X and Y coordinates for 17 different body joints (like elbows, knees, shoulders), plus a probability score of whether that joint is visible.

Fer collumn:
- List of dictionaries (one for each detected person). 
- Dictionaries keys: 
    - 'bbox': A list of 4 numbers [x1, y1, x2, y2] for the box around just the face.  
    - 'top_emotion': A string (like 'Neutral', 'Anger', 'Happiness').  
    - 'probabilities': A dictionary containing the exact percentage scores for all 8 possible emotions.  
    - 'landmarks': A giant list of 106 X/Y coordinates mapping out the eyes, nose, mouth, and jawline.

### Initial Data Exploration

In [59]:
# check for standard Pandas nulls (NaN)
print("Standard Missing Values:")
print(data.isnull().sum())

print('\n Data info')

data.info()



Standard Missing Values:
Frame    0
Poses    0
Fer      0
dtype: int64

 Data info
<class 'pandas.DataFrame'>
RangeIndex: 2110 entries, 0 to 2109
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Frame   2110 non-null   str   
 1   Poses   2110 non-null   object
 2   Fer     2110 non-null   object
dtypes: object(2), str(1)
memory usage: 153.7+ KB


### Checking for missing values

In [69]:
poses = data['Poses']
faces = data['Fer']
empty_poses =0
empty_faces = 0
for i in range(len(poses)):
    if isinstance(poses[i], list):
        if len(poses[i]) == 0:
            empty_poses += 1
    else:
        empty_poses += 1

    if isinstance(faces[i], list):
        if len(faces[i]) == 0:
            empty_faces += 1
    else:
        empty_faces += 1

print(f"Frames with 0 human poses detected: {empty_poses}")
print(f"Frames with 0 faces detected: {empty_faces}")

Frames with 0 human poses detected: 7
Frames with 0 faces detected: 8


In [ ]:
# Create a new column for the number of people per frame
data['People_count'] = data['Poses'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Get the summary distribution
print("Distribution of people detected per frame:")
print(data['People_count'].value_counts().sort_index())

